# 09. Engagement Twitter

Ce notebook exploite les métriques d'engagement (retweets, likes, replies, quotes) des tweets du corpus pour tester si **la radicalité discursive paie** en termes de visibilité.

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from config import CORPUS_V3, BLOC_ORDER, BLOC_COLORS

FIG_DIR = Path("../figures")
FIG_DIR.mkdir(exist_ok=True)

def save(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.close()

## Chargement : tweets uniquement

Les interventions AN ont engagement = 0. On filtre sur `arena == "Twitter"`.

In [ ]:
df = pd.read_parquet(CORPUS_V3)
eng_col = "engagement" if "engagement" in df.columns else None
if eng_col is None:
    if all(c in df.columns for c in ["retweets", "likes", "replies", "quotes"]):
        df["engagement"] = df["retweets"].fillna(0) + df["likes"].fillna(0) + df["replies"].fillna(0) + df["quotes"].fillna(0)
        eng_col = "engagement"

df_tw = df[(df.get("arena", "") == "Twitter") & df["bloc"].isin(BLOC_ORDER)].copy()
if eng_col and len(df_tw) > 0:
    df_tw["log_engagement"] = np.log1p(df_tw[eng_col])
    df_tw["abs_stance"] = df_tw["stance_v3"].abs()
    print(f"Tweets : {len(df_tw):,} | engagement médian = {df_tw[eng_col].median():.0f}")
else:
    print("Colonnes engagement absentes ou corpus Twitter vide.")

## Fig 50 : Engagement par bloc

In [ ]:
if eng_col and len(df_tw) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    med = df_tw.groupby("bloc")[eng_col].median().reindex(BLOC_ORDER)
    ax.bar(range(len(BLOC_ORDER)), med.values, color=[BLOC_COLORS[b] for b in BLOC_ORDER])
    ax.set_xticks(range(len(BLOC_ORDER)))
    ax.set_xticklabels([b.replace(" ", "\n") for b in BLOC_ORDER])
    ax.set_ylabel("Engagement médian (rt + likes + replies + quotes)")
    ax.set_title("Engagement Twitter par bloc politique")
    save("fig50_engagement_par_bloc")

## Question : La radicalité paie-t-elle ?

Régression : log(1 + engagement) ~ |stance| + bloc (dummies).

In [ ]:
if eng_col and len(df_tw) > 100:
    X = pd.get_dummies(df_tw[["abs_stance", "bloc"]], columns=["bloc"], drop_first=True)
    X = sm.add_constant(X)
    y = df_tw["log_engagement"]
    mod = sm.OLS(y, X).fit(cov_type="HC3")
    print(mod.summary2().tables[1])
    coef_abs = mod.params.get("abs_stance", np.nan)
    print(f"\n→ Coefficient |stance| : {coef_abs:.4f} (positif = la radicalité augmente l'engagement)")

## Fig 51 : Engagement vs |stance| par bloc

In [ ]:
if eng_col and len(df_tw) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    for bloc in BLOC_ORDER:
        sub = df_tw[df_tw["bloc"] == bloc]
        ax.scatter(sub["abs_stance"], sub["log_engagement"], alpha=0.2, s=10, color=BLOC_COLORS[bloc], label=bloc)
    ax.set_xlabel("|Stance| (radicalité)")
    ax.set_ylabel("log(1 + engagement)")
    ax.legend()
    ax.set_title("Engagement vs radicalité discursive par bloc")
    save("fig51_engagement_vs_radicalite")